# 05 · GRPO training

Companion notebook for [Chapter 6](../chapters/ch06-grpo.md).

**You will:** inspect a single GRPO step, then run a short training loop and watch
the two numbers that matter — mean reward and mean response length.

!!! warning "GPU required for the training cells"

    The single-step inspection works on a CPU with small `max_new_tokens`. A real
    run does not.

## 0 · Setup

Run this cell first. It picks a device and imports everything the notebook needs.

If you get `ModuleNotFoundError: reasoning_from_scratch`, your kernel is not the
course environment — see the [setup page](../start/setup.md).

In [ ]:
import torch
from reasoning_from_scratch.ch02 import get_device

device = get_device()
print("torch", torch.__version__)

In [ ]:
from pathlib import Path
from reasoning_from_scratch.qwen3 import (
    Qwen3Model, QWEN_CONFIG_06_B, Qwen3Tokenizer, download_qwen3_small,
)

download_qwen3_small(kind="base", out_dir="qwen3")     # 1.5 GB, cached after the first run

tokenizer = Qwen3Tokenizer(
    tokenizer_file_path=Path("qwen3") / "tokenizer-base.json"
)
model = Qwen3Model(QWEN_CONFIG_06_B)
model.load_state_dict(torch.load(Path("qwen3") / "qwen3-0.6B-base.pth"))
model.to(device)
print("parameters:", sum(p.numel() for p in model.parameters()) / 1e9, "B")

## 1 · The reward is binary, and strict

Before any training, check what earns reward. Ground truth is `83`.

In [ ]:
from reasoning_from_scratch.ch06 import reward_rlvr

rollouts = [
    r"\boxed{83}",
    r"The correct answer is \boxed{83}",
    r"The final answer is 83",      # right value, no box
    r"We get \boxed{38}",          # right format, wrong value
]
for r in rollouts:
    print(f"{reward_rlvr(r, '83')}  {r!r}")

**The third row is the whole course in one line.** The model can already answer
correctly and still earn nothing, because it has not learned the format. Most of
the early accuracy jump is it learning to box.

## 2 · The training set

In [ ]:
from reasoning_from_scratch.ch06 import load_math_train

math_train = load_math_train()      # 12,000 problems: MATH minus MATH-500
print("problems:", len(math_train))
print("fields  :", list(math_train[0].keys()))
print()
print(math_train[4]["problem"][:300])
print("answer:", math_train[4]["answer"])

## 3 · One GRPO step, inspected

**Predict first:** with `num_rollouts=2`, what are the possible values of
`advantages`?

In [ ]:
from reasoning_from_scratch.ch06 import compute_grpo_loss

stats = compute_grpo_loss(
    model, tokenizer, math_train[0], device,
    num_rollouts=2, max_new_tokens=128, temperature=0.8, top_p=0.9,
)
print("rewards    :", stats["rewards"])
print("advantages :", [round(a, 3) for a in stats["advantages"]])
print("loss       :", round(stats["loss"], 4))
print("gen lengths:", [s["gen_len"] for s in stats["samples"]])
print()
print(stats["samples"][0]["text"][:500])

**The answer to the prediction:** with two rollouts the advantages are either
`[0, 0]` (both same reward — no learning signal) or `[-1, 1]`/`[1, -1]` (mixed).
Two thirds of your steps teach nothing. Real runs use more rollouts for exactly
this reason.

## 4 · Run 20 steps

**Predict before running:** what happens to `avg_response_len` over 20 steps?

In [ ]:
model = train_rlvr_grpo(
    model, tokenizer, math_train, device,
    steps=20, num_rollouts=2, max_new_tokens=128, lr=1e-5,
    checkpoint_every=10, checkpoint_dir="checkpoints",
    csv_log_path="grpo_metrics.csv",
)

In [ ]:
import pandas as pd
df = pd.read_csv("grpo_metrics.csv")
print(df.to_string(index=False))
print()
print("mean reward  :", round(df["reward_avg"].mean(), 3))
print("mean length  :", round(df["avg_response_len"].mean(), 1), "tokens")
print("length trend :", round(df["avg_response_len"].iloc[-5:].mean() - df["avg_response_len"].iloc[:5].mean(), 1))

## 5 · What the reference run looks like

From the repo's 500-step log:

| Step | Mean reward | Mean response length | MATH-500 |
| ---: | ---: | ---: | ---: |
| 1 | 0.000 | 6.4 tokens | 15.6% |
| 2 | 0.125 | 6.6 tokens | — |
| 500 | 0.125 | 762 tokens | 33.0% |

Peak accuracy **46.2% at step 50**, then decay to 33.0%. Reward keeps climbing;
correctness does not. That divergence is [chapter 7](../chapters/ch07-reading-runs.md).

20 steps will not show the full arc. What you can look for: does length grow at
all, and does reward move off zero?

## 6 · Evaluate what you trained

```python
from reasoning_from_scratch.ch03 import evaluate_math500_stream, load_math500_test
math500 = load_math500_test()
evaluate_math500_stream(model, tokenizer, device, math_data=math500[:50])
```

Compare against the 15.6% baseline. A 20-step run should not be expected to beat
it — you are checking the plumbing.

## 7 · Exercises

1. Print `rewards` and `advantages` for 20 steps. What fraction of groups were
   all-identical (zero gradient)?
2. Set `num_rollouts=4` and re-run 20 steps. Faster reward growth? What did it
   cost in wall-clock?
3. Change the reward to `fallback="number_then_full"` and run 20 steps. Does
   formatting collapse?
4. Run 100 steps and plot `avg_response_len`. Where does the blow-up start?